# EDA de Titanic con DuckDB

Cuaderno instructivo **SQL-first**: describimos el conjunto original sin limpiar, imputar ni eliminar filas. Las consultas devuelven las claves y las columnas comunes `total` (pasajeros del grupo), `supervivientes` y `tasa`. Una tasa es una descripción de asociación en estos datos; **no demuestra causalidad**.

## 1. Conexión y carga

DuckDB permite consultar CSV directamente con SQL. `read_csv_auto` infiere el esquema y crea una tabla temporal en memoria; no modifica el fichero original. La búsqueda de la ruta parte de la ubicación habitual del cuaderno y también contempla ejecuciones desde la raíz del repositorio.

In [1]:
from pathlib import Path
from hashlib import sha256
import duckdb

candidatas = [
    Path('titanic.csv'),
    Path('../05-recursos/datasets/titanic.csv'),
    Path.cwd() / '../05-recursos/datasets/titanic.csv',
    Path.cwd() / '02-tratamiento-datos/05-recursos/datasets/titanic.csv',
]
for padre in Path.cwd().parents:
    candidatas.append(padre / '02-tratamiento-datos/05-recursos/datasets/titanic.csv')
CSV = next((ruta.resolve() for ruta in candidatas if ruta.exists()), None)
if CSV is None:
    raise FileNotFoundError('No se encontró el CSV original de Titanic en ../05-recursos/datasets/titanic.csv')

con = duckdb.connect()
con.execute("CREATE OR REPLACE TEMP TABLE titanic AS SELECT * FROM read_csv_auto(?)", [str(CSV)])
print(f'CSV: {CSV}')
HUELLA_CSV = sha256(CSV.read_bytes()).hexdigest()
print('SHA-256:', HUELLA_CSV)


CSV: /datos/RafaelAlberti/RafaelAlberti26_27/Modulos/PIA/02-tratamiento-datos/05-recursos/datasets/titanic.csv
SHA-256: 8331e5a2532f5fdb66153a8fc053df08a7dc9d2584f246c7fb1752f0a7139ca5


## 2. Tamaño, esquema y nulos

`COUNT(*)` cuenta filas; `DESCRIBE` muestra el esquema; `COUNT(*) - COUNT(columna)` obtiene los nulos de cada columna sin alterar los datos.

In [2]:
shape = con.execute('SELECT COUNT(*) AS filas, COUNT(*) FILTER (WHERE TRUE) AS filas_verificacion FROM titanic').fetchall()[0]
esquema = con.sql('DESCRIBE titanic').df()
nulos = con.sql('''
SELECT
  COUNT(*) - COUNT(Age) AS Age,
  COUNT(*) - COUNT(Cabin) AS Cabin,
  COUNT(*) - COUNT(Embarked) AS Embarked
FROM titanic
''').df()
print(f'shape: ({shape[0]}, {esquema.shape[0]})')
esquema


shape: (891, 12)


,column_name,column_type,null,key,default,extra
0,PassengerId,BIGINT,YES,None,None,None
1,Survived,BIGINT,YES,None,None,None
2,Pclass,BIGINT,YES,None,None,None
3,Name,VARCHAR,YES,None,None,None
4,Sex,VARCHAR,YES,None,None,None
5,Age,DOUBLE,YES,None,None,None
6,SibSp,BIGINT,YES,None,None,None
7,Parch,BIGINT,YES,None,None,None
8,Ticket,VARCHAR,YES,None,None,None
9,Fare,DOUBLE,YES,None,None,None


In [3]:
assert shape[0] == 891 and esquema.shape[0] == 12
assert list(esquema['column_name']) == ['PassengerId','Survived','Pclass','Name','Sex','Age','SibSp','Parch','Ticket','Fare','Cabin','Embarked']
assert nulos.iloc[0].to_dict() == {'Age': 177, 'Cabin': 687, 'Embarked': 2}
print('Nulos:')
print(nulos.to_string(index=False))


Nulos:
 Age  Cabin  Embarked
 177    687         2


## 3. Función descriptiva en SQL

En cada consulta, `COUNT(*) AS total` cuenta los pasajeros del grupo; `SUM(Survived) AS supervivientes` cuenta quienes sobrevivieron porque Survived vale 0/1. La tasa divide supervivientes por ese mismo total del grupo. No se repite 891 ni 714 como denominador en todas las filas.

In [4]:
def mostrar_sql(sql):
    resultado = con.sql(sql).df()
    display(resultado)
    return resultado

global_eda = mostrar_sql('''
SELECT COUNT(*) AS total,
       SUM(Survived) AS supervivientes,
       SUM(Survived)::DOUBLE / COUNT(*) AS tasa
FROM titanic
''')
assert global_eda.loc[0, ['total','supervivientes']].astype(int).tolist() == [891,342]
assert abs(global_eda.loc[0, 'tasa'] - 342/891) < 1e-6


,total,supervivientes,tasa
0,891,342.0,0.383838


## 4. Supervivencia por sexo

El filtro `WHERE Sex IN (...)` hace explícitas las categorías solicitadas y `GROUP BY Sex` resume cada una.

In [5]:
por_sexo = mostrar_sql('''
SELECT Sex, COUNT(*) AS total,
       SUM(Survived) AS supervivientes,
       SUM(Survived)::DOUBLE / COUNT(*) AS tasa
FROM titanic
WHERE Sex IN ('female', 'male')
GROUP BY Sex ORDER BY CASE Sex WHEN 'female' THEN 1 ELSE 2 END
''')
assert por_sexo[['total','supervivientes']].astype(int).values.tolist() == [[314,233],[577,109]]
assert all(abs(a-b) < 1e-6 for a,b in zip(por_sexo.tasa, [233/314,109/577]))


,Sex,total,supervivientes,tasa
0,female,314,233.0,0.742038
1,male,577,109.0,0.188908


## 5. Supervivencia por clase

`Pclass` es ordinal, por eso ordenamos ascendentemente para leer las clases 1, 2 y 3.

In [6]:
por_clase = mostrar_sql('''
SELECT Pclass, COUNT(*) AS total,
       SUM(Survived) AS supervivientes,
       SUM(Survived)::DOUBLE / COUNT(*) AS tasa
FROM titanic WHERE Pclass IN (1,2,3)
GROUP BY Pclass ORDER BY Pclass
''')
assert por_clase[['total','supervivientes']].astype(int).values.tolist() == [[216,136],[184,87],[491,119]]
assert all(abs(a-b) < 1e-6 for a,b in zip(por_clase.tasa, [136/216,87/184,119/491]))


,Pclass,total,supervivientes,tasa
0,1,216,136.0,0.629630
1,2,184,87.0,0.472826
2,3,491,119.0,0.242363


## 6. Cruce de clase y sexo

Agrupamos por las dos dimensiones y ordenamos primero `Pclass` y después `Sex`, manteniendo `female` antes de `male`.

In [7]:
por_clase_sexo = mostrar_sql('''
SELECT Pclass, Sex, COUNT(*) AS total,
       SUM(Survived) AS supervivientes,
       SUM(Survived)::DOUBLE / COUNT(*) AS tasa
FROM titanic
WHERE Pclass IN (1,2,3) AND Sex IN ('female','male')
GROUP BY Pclass, Sex
ORDER BY Pclass, CASE Sex WHEN 'female' THEN 1 ELSE 2 END
''')
assert por_clase_sexo[['total','supervivientes']].astype(int).values.tolist() == [[94,91],[122,45],[76,70],[108,17],[144,72],[347,47]]
assert all(abs(a-b) < 1e-6 for a,b in zip(por_clase_sexo.tasa, [91/94,45/122,70/76,17/108,72/144,47/347]))


,Pclass,Sex,total,supervivientes,tasa
0,1,female,94,91.0,0.968085
1,1,male,122,45.0,0.368852
2,2,female,76,70.0,0.921053
3,2,male,108,17.0,0.157407
4,3,female,144,72.0,0.500000
5,3,male,347,47.0,0.135447


## 7. Edad: menores y adultos

Aquí sí excluimos únicamente los registros cuyo `Age` es nulo, porque no pueden clasificarse en `< 18` o `>= 18`. No imputamos edades ni eliminamos filas de la tabla original.

In [8]:
por_edad = mostrar_sql('''
SELECT CASE WHEN Age < 18 THEN '<18' ELSE '>=18' END AS grupo_edad,
       COUNT(*) AS total,
       SUM(Survived) AS supervivientes,
       SUM(Survived)::DOUBLE / COUNT(*) AS tasa
FROM titanic
WHERE Age IS NOT NULL
GROUP BY grupo_edad
ORDER BY CASE grupo_edad WHEN '<18' THEN 1 ELSE 2 END
''')
assert por_edad[['total','supervivientes']].astype(int).values.tolist() == [[113,61],[601,229]]
assert all(abs(a-b) < 1e-6 for a,b in zip(por_edad.tasa, [61/113,229/601]))
assert int(por_edad['total'].sum()) == 714
for tabla_, claves in [(global_eda, []), (por_sexo, ['Sex']), (por_clase, ['Pclass']), (por_clase_sexo, ['Pclass', 'Sex']), (por_edad, ['grupo_edad'])]:
    assert list(tabla_.columns) == claves + ['total', 'supervivientes', 'tasa']
assert por_sexo['Sex'].tolist() == ['female', 'male']
assert por_edad['grupo_edad'].tolist() == ['<18', '>=18']
print('Todas las validaciones han pasado.')


,grupo_edad,total,supervivientes,tasa
0,<18,113,61.0,0.539823
1,>=18,601,229.0,0.381032


Todas las validaciones han pasado.


## Cierre

Hemos descrito asociaciones observadas en el Titanic original mediante SQL. Estas diferencias de tasas no permiten concluir que el sexo, la clase o la edad causaran el resultado: harían falta un diseño causal y supuestos adicionales. No se han aplicado limpieza, imputación, eliminación de filas, gráficos, modelos ni benchmarks.

## Pregunta propia del equipo experto

Escribid una pregunta adicional y resolvedla con una consulta SQL: filtro, agrupación y columnas total/supervivientes/tasa. Explicad qué filas forman el denominador y qué pasa con los valores ausentes. No repitáis el núcleo ni modifiquéis la tabla original.

**Pregunta:** por completar.
**Filtro, población y nulos:** por completar.
**Interpretación y limitación:** por completar.

Entrega experta y mixta: [entrega-comparativa-motores-titanic.md](entrega-comparativa-motores-titanic.md).

In [9]:
# TODO: escribid vuestra consulta y ejecutadla con mostrar_sql(...).
# COUNT(*) AS total, SUM(Survived) AS supervivientes y su cociente.
